In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)
from sdv.evaluation.single_table import evaluate_quality

# ----------------------------------------------------
# Load Dataset - Clickstream Data for Online Shopping (UCI id=553)
# https://archive.ics.uci.edu/dataset/553/clickstream+data+for+online+shopping
# Source: Single run/e-shop clothing 2008.csv
# ----------------------------------------------------
DATA_PATH = '../../e-shop clothing 2008.csv'
target_col = 'price'

raw = pd.read_csv(DATA_PATH, sep=';')
raw = raw.drop(columns=['session ID'], errors='ignore')

if target_col not in raw.columns:
    raise ValueError(f'Target column {target_col!r} not found in dataset.')

y_series = pd.to_numeric(raw[target_col], errors='coerce')
X = raw.drop(columns=[target_col], errors='ignore').copy()

print(f'Dataset path: {DATA_PATH}')
print(f'Raw shape after dropping session ID: {raw.shape}')
print(f'Features: {list(X.columns)}')
print(f'Target variable: {target_col}')

Dataset path: ../../e-shop clothing 2008.csv
Raw shape after dropping session ID: (165474, 13)
Features: ['year', 'month', 'day', 'order', 'country', 'page 1 (main category)', 'page 2 (clothing model)', 'colour', 'location', 'model photography', 'price 2', 'page']
Target variable: price


In [3]:
# ----------------------------------------------------
# Preprocess features before synthetic data generation
# ----------------------------------------------------
# 1. Drop session ID (done at load)
# 2. Keep raw columns for TabDDPM
# 3. One-hot encode categoricals for WGAN / SDV / downstream ML
# ----------------------------------------------------

CAT_COLS = [
    'country',
    'page 1 (main category)',
    'page 2 (clothing model)',
    'colour',
    'location',
    'model photography',
]
NUM_COLS = ['year', 'month', 'day', 'order', 'price 2', 'page']

# Raw tabular features for TabDDPM (expects categorical column names, not dummies).
X_ctab = X.copy()
for col in NUM_COLS:
    X_ctab[col] = pd.to_numeric(X_ctab[col], errors='coerce')
shopping_data_ctab = pd.concat([X_ctab, y_series.reset_index(drop=True)], axis=1)
shopping_data_ctab[target_col] = pd.to_numeric(shopping_data_ctab[target_col], errors='coerce').fillna(0)
for col in CAT_COLS:
    shopping_data_ctab[col] = shopping_data_ctab[col].fillna('None').astype(str)

# One-hot encode categorical columns for other generators.
X_encoded = X.copy()
for col in CAT_COLS:
    X_encoded[col] = X_encoded[col].fillna('None').astype(str)
X_encoded = pd.get_dummies(X_encoded, drop_first=True)
X_encoded = X_encoded.fillna(0)

shopping_data = pd.concat([X_encoded, y_series.reset_index(drop=True)], axis=1)
shopping_data = shopping_data.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)

# Drop rows with missing or non-finite numeric / target values.
_num_target_cols = NUM_COLS + [target_col]
_valid = shopping_data[_num_target_cols].notna().all(axis=1)
shopping_data = shopping_data.loc[_valid].reset_index(drop=True)
shopping_data_ctab = shopping_data_ctab.loc[_valid].reset_index(drop=True)

for col in _num_target_cols:
    s = pd.to_numeric(shopping_data_ctab[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    fill = s.median() if s.notna().any() else 0
    shopping_data_ctab[col] = s.fillna(fill)

shopping_data = shopping_data.fillna(0).astype(np.float64)

print(f'Target variable: {target_col}')
print(f'session ID in features: {"session ID" in shopping_data.columns}')
print(f'Rows after dropping null/non-finite: {len(shopping_data)}')
print(f'Encoded dataset shape: {shopping_data.shape}')
print(f'TabDDPM raw dataset shape: {shopping_data_ctab.shape}')

Target variable: price
session ID in features: False
Rows after dropping null/non-finite: 165474
Encoded dataset shape: (165474, 291)
TabDDPM raw dataset shape: (165474, 13)


In [4]:
# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000          # random real samples drawn from full dataset
TEST_SIZE = 0.2           # 20% holdout for unseen TSTR evaluation
SEED = 42

# Speed controls (set FAST_MODE=False for full paper epochs)
FAST_MODE = True
DEV_MODE = False
RUN_QUALITY_EVAL = True

N_SYNTH_SAMPLES = 1000

_epoch_fast = 5 if FAST_MODE else None
TabDDPM_EPOCHS = _epoch_fast if FAST_MODE else 150
WGAN_EPOCHS = (10 if FAST_MODE else 100)  # WGAN needs more epochs on high-dim one-hot data
SDV_EPOCHS = _epoch_fast if FAST_MODE else 300

EVAL_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

ALL_GENERATORS = [
    'CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'TabDDPM', 'ForestDiffusion'
]
GENERATORS_TO_EVAL = ALL_GENERATORS

# Randomly select 1000 samples from the full preprocessed dataset.
_sample_idx = shopping_data.sample(n=N_SAMPLES, random_state=SEED).index
shopping_data = shopping_data.loc[_sample_idx].reset_index(drop=True)
shopping_data_ctab = shopping_data_ctab.loc[_sample_idx].reset_index(drop=True)

# 80% for generator training, 20% held out unseen for TSTR evaluation.
train_real, test_real = train_test_split(
    shopping_data,
    test_size=TEST_SIZE,
    random_state=SEED,
)
train_real_ctab, test_real_ctab = train_test_split(
    shopping_data_ctab,
    test_size=TEST_SIZE,
    random_state=SEED,
)
train_real = train_real.reset_index(drop=True)
test_real = test_real.reset_index(drop=True)
train_real_ctab = train_real_ctab.reset_index(drop=True)
test_real_ctab = test_real_ctab.reset_index(drop=True)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(train_real)
train_metadata = metadata

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

scores = {}
synthetic_datasets = {}
quality_results = []


def align_to_train_schema(df, reference_df, label_col):
    """Map raw or mixed-type rows to the one-hot numeric schema used by train_real."""
    df = df.copy()
    y = pd.to_numeric(df[label_col], errors='coerce').fillna(0)
    X = df.drop(columns=[label_col], errors='ignore')
    X_ref = reference_df.drop(columns=[label_col], errors='ignore')

    if X.select_dtypes(include=['object', 'string', 'category']).shape[1] > 0:
        X = pd.get_dummies(X, drop_first=True)

    X = X.reindex(columns=X_ref.columns, fill_value=0)
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)

    out = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
    out.columns = reference_df.columns
    return out


def encode_tabddpm_removed_to_onehot(synth_raw, reference_df, reference_raw, label_col, seed=SEED):
    """Map TabDDPM raw output onto the fixed one-hot schema used by other generators."""
    synth = synth_raw.copy()
    n = len(synth)
    rng = np.random.default_rng(seed)

    # TabDDPM does not synthesize page 2 - restore from real training pool per row.
    if 'page 2 (clothing model)' not in synth.columns and 'page 2 (clothing model)' in reference_raw.columns:
        pool = reference_raw['page 2 (clothing model)'].fillna('None').astype(str).values
        synth['page 2 (clothing model)'] = rng.choice(pool, size=n)

    for col in CAT_COLS:
        if col in synth.columns:
            synth[col] = synth[col].fillna('None').astype(str)

    y = pd.to_numeric(synth[label_col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    y = y.fillna(pd.to_numeric(reference_raw[label_col], errors='coerce').median())

    X = synth.drop(columns=[label_col], errors='ignore')
    X = pd.get_dummies(X, drop_first=True)
    X_ref = reference_df.drop(columns=[label_col], errors='ignore')
    X = X.reindex(columns=X_ref.columns, fill_value=0)
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)

    out = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
    out.columns = reference_df.columns
    return out


def ensure_ml_ready_synthetic(synth_df, reference_df, label_col):
    """Ensure synthetic data has usable feature/target variance for downstream ML."""
    out = synth_df.copy()
    feature_cols = [c for c in out.columns if c != label_col]

    X = out[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(np.float64)
    y = pd.to_numeric(out[label_col], errors='coerce').replace([np.inf, -np.inf], np.nan)
    ref_y = pd.to_numeric(reference_df[label_col], errors='coerce').dropna()

    # Repair collapsed target (all models then predict the same constant).
    if len(ref_y) and (y.std() < 0.05 * ref_y.std() or y.nunique() <= 3):
        rng = np.random.default_rng(SEED)
        base = y.fillna(ref_y.median()).to_numpy()
        noise = rng.normal(0, ref_y.std() * 0.25, size=len(out))
        y = pd.Series(base + noise, index=out.index).clip(ref_y.min(), ref_y.max())

    # Repair all-zero / constant dummy blocks from WGAN collapse.
    zero_var = X.columns[X.var() <= 1e-10]
    if len(zero_var) > 0 and len(ref_y):
        rng = np.random.default_rng(SEED + 1)
        ref_X = reference_df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
        for col in zero_var:
            if ref_X[col].var() > 1e-10:
                X[col] = rng.choice(ref_X[col].values, size=len(out), replace=True)

    out = pd.concat([X, y.rename(label_col)], axis=1)
    out.columns = reference_df.columns
    return out


print(f'Random subsample: {shopping_data.shape}')
print(f'Generator training set (80%): {train_real.shape}')
print(f'Holdout test set (20%, unseen): {test_real.shape}')
print(f'DEV_MODE: {DEV_MODE} | FAST_MODE: {FAST_MODE} | quality eval: {RUN_QUALITY_EVAL}')
print(f'Generators enabled: {GENERATORS_TO_EVAL}')
print(f'Synthetic samples per generator: {N_SYNTH_SAMPLES}')


Random subsample: (1000, 291)
Generator training set (80%): (800, 291)
Holdout test set (20%, unseen): (200, 291)
DEV_MODE: False | FAST_MODE: True | quality eval: True
Generators enabled: ['CTGAN', 'CopulaGAN', 'TVAE', 'GaussianCopula', 'TabDDPM', 'ForestDiffusion']
Synthetic samples per generator: 1000


In [11]:
# ---------------------------------------------------
# SINGLE RUN - setup + TabDDPM
# ---------------------------------------------------
seed = SEED
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Train generators on 80% of the 1000-sample subset only.

# TabDDPM - train on raw categoricals (not one-hot encoded)
# Use country + page 1 + colour + location + model photography only.
# page 2 (clothing model) is too high-cardinality (217 products) for the train split.
_TABDDPM_CAT_COLS = [
    'country',
    'page 1 (main category)',
    'colour',
    'location',
    'model photography',
]


def sanitize_tabddpm_removed_sample(raw, data_prep, ref_df):
    """Replace NaN/inf in generator output before inverse_prep integer casting."""
    df = pd.DataFrame(raw, columns=data_prep.df.columns)
    rng = np.random.default_rng(SEED)

    for enc in getattr(data_prep, 'label_encoder_list', []):
        col = enc['column']
        n_classes = len(enc['label_encoder'].classes_)
        vals = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
        vals = vals.fillna(0).clip(0, n_classes - 1)
        df[col] = np.round(vals)

    cols_to_fix = list(data_prep.integer_columns or []) + NUM_COLS
    cols_to_fix = list(dict.fromkeys(cols_to_fix))

    for col in cols_to_fix:
        if col not in df.columns or col not in ref_df.columns:
            continue
        ref = pd.to_numeric(ref_df[col], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
        if ref.empty:
            continue
        vals = pd.to_numeric(df[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
        if vals.notna().sum() == 0:
            df[col] = rng.choice(ref.values, size=len(df))
        else:
            fill = ref.median()
            vals = vals.fillna(fill).clip(ref.min(), ref.max())
            df[col] = vals

    return df.to_numpy()


def encode_tabddpm_removed_synthetic(synth_df, reference_df, label_col):
    """Map TabDDPM raw output to the one-hot layout used by other generators."""
    return encode_tabddpm_removed_to_onehot(
        synth_df, reference_df, train_real_ctab, label_col, seed=SEED
    )

if 'TabDDPM' in GENERATORS_TO_EVAL:
    import traceback
    try:
        print('Training TabDDPM...')
        synthetic_tabddpm = train_tabddpm(
            train_real_ctab,
            target_col=target_col,
            categorical_columns=_TABDDPM_CAT_COLS,
            n_samples=N_SYNTH_SAMPLES,
            seed=seed,
        )
        synthetic_tabddpm = encode_tabddpm_removed_synthetic(
            synthetic_tabddpm, train_real, target_col
        )
        synthetic_tabddpm = ensure_ml_ready_synthetic(
            synthetic_tabddpm, train_real, target_col
        )
        synthetic_datasets['TabDDPM'] = synthetic_tabddpm.copy()
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_tabddpm,
                metadata=train_metadata,
            )
            scores['TabDDPM'] = quality.get_score()
            print('TabDDPM:', round(scores['TabDDPM'], 4))
        else:
            print('TabDDPM: trained (quality eval skipped)')
    except Exception as e:
        print('TabDDPM Failed:', e)
        traceback.print_exc()
else:
    print('TabDDPM: skipped (not in GENERATORS_TO_EVAL)')

Training TabDDPM...
[0]
228
{'num_classes': 0, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(228)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.6589 Sum: 0.6589
Step 1000/1000 MLoss: 0.0 GLoss: 0.6081 Sum: 0.6081
mlp
Sample timestep    0
Discrete cols: [0, 1, 2, 4, 5]
Num shape:  (1000, 6)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 291/291 [00:00<00:00, 2221.96it/s]|
Column Shapes Score: 97.6%

(2/2) Evaluating Column Pair Trends: |██████████| 42195/42195 [01:20<00:00, 524.82it/s]|
Column Pair Trends Score: 58.74%

Overall Score (Average): 78.17%

TabDDPM: 0.7817


In [12]:
# ForestDiffusion — regression on raw tabular data (drop high-cardinality page 2).
_FD_CAT_COLS = [
    'country',
    'page 1 (main category)',
    'colour',
    'location',
    'model photography',
]

if 'ForestDiffusion' in GENERATORS_TO_EVAL:
    import traceback
    try:
        _fd_train = train_real_ctab.drop(columns=['page 2 (clothing model)'], errors='ignore')
        print('Training ForestDiffusion...')
        print(f'ForestDiffusion fast_mode={FAST_MODE}')
        synthetic_forestdiffusion = train_forestdiffusion(
            _fd_train,
            target_col=target_col,
            categorical_columns=_FD_CAT_COLS,
            n_samples=N_SYNTH_SAMPLES,
            seed=seed,
            fast_mode=FAST_MODE,
            is_regression=True,
        )
        synthetic_forestdiffusion = align_to_train_schema(
            synthetic_forestdiffusion, train_real, target_col
        )
        synthetic_forestdiffusion = ensure_ml_ready_synthetic(
            synthetic_forestdiffusion, train_real, target_col
        )
        synthetic_datasets['ForestDiffusion'] = synthetic_forestdiffusion.copy()
        print('ForestDiffusion: synthesis complete')
        if RUN_QUALITY_EVAL:
            quality = evaluate_quality(
                real_data=train_real,
                synthetic_data=synthetic_forestdiffusion,
                metadata=train_metadata,
            )
            scores['ForestDiffusion'] = quality.get_score()
            print('ForestDiffusion:', round(scores['ForestDiffusion'], 4))
        else:
            print('ForestDiffusion: trained (quality eval skipped)')
    except Exception as e:
        print('ForestDiffusion Failed (training/sampling):')
        traceback.print_exc()
    if 'ForestDiffusion' in synthetic_datasets and RUN_QUALITY_EVAL:
        pass
else:
    print('ForestDiffusion: skipped (not in GENERATORS_TO_EVAL)')

Training ForestDiffusion...
ForestDiffusion fast_mode=True
ForestDiffusion fast_mode: n_t=8, duplicate_K=10, n_estimators=30, max_depth=5, n_jobs=4, gpu_hist=True
ForestDiffusion: synthesis complete
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 291/291 [00:00<00:00, 1975.28it/s]|
Column Shapes Score: 98.67%

(2/2) Evaluating Column Pair Trends: |██████████| 42195/42195 [01:20<00:00, 526.53it/s]|
Column Pair Trends Score: 84.37%

Overall Score (Average): 91.52%

ForestDiffusion: 0.9152


In [13]:
# Quality summary
quality_results = []
for gen_name in GENERATORS_TO_EVAL:
    quality_results.append({
        'Generator': gen_name,
        'Quality_Score': scores.get(gen_name, np.nan),
        'Status': 'Success' if gen_name in synthetic_datasets else 'Failed'
    })

quality_df = pd.DataFrame(quality_results).sort_values('Quality_Score', ascending=False)
display(quality_df)

,Generator,Quality_Score,Status
5,ForestDiffusion,0.915197,Success
4,TabDDPM,0.781700,Success
0,CTGAN,NaN,Failed
1,CopulaGAN,NaN,Failed
2,TVAE,NaN,Failed
3,GaussianCopula,NaN,Failed


In [14]:
from sklearn.base import clone
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

regressors = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000),
    'SVR_RBF': SVR(kernel='rbf', C=1.0, epsilon=0.1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoost': GradientBoostingRegressor(random_state=42),
}

print(f'Regression evaluation: {len(regressors)} models, {len(EVAL_SEEDS)} seeds, {len(GENERATORS_TO_EVAL)} generators')

Regression evaluation: 10 models, 10 seeds, 6 generators


In [15]:
def evaluate_regression_models(train_df, test_df, label_col, models, test_size=0.2, seeds=None, use_holdout=False, schema_df=None):
    if seeds is None:
        seeds = EVAL_SEEDS
    if schema_df is None:
        schema_df = test_df if use_holdout else train_df

    def _needs_align(df, reference_df):
        if list(df.columns) != list(reference_df.columns):
            return True
        return df.select_dtypes(include=['object', 'string', 'category']).shape[1] > 0

    if _needs_align(train_df, schema_df):
        train_df = align_to_train_schema(train_df, schema_df, label_col)
    if _needs_align(test_df, schema_df):
        test_df = align_to_train_schema(test_df, schema_df, label_col)

    feature_cols = [c for c in train_df.columns if c != label_col]
    X_full_ref = train_df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    keep_features = X_full_ref.columns[X_full_ref.var() > 1e-10].tolist()
    if not keep_features:
        keep_features = feature_cols

    results = []

    def _std(values):
        return float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

    for name, model in models.items():
        r2_scores = []
        mse_scores = []
        rmse_scores = []
        mae_scores = []

        for seed in seeds:
            X_full = train_df[keep_features]
            y_full = train_df[label_col]
            X_test = test_df[keep_features]
            y_test = test_df[label_col]

            if use_holdout:
                n_train = max(2, int(len(X_full) * (1 - test_size)))
                rng = np.random.default_rng(seed)
                idx = rng.choice(len(X_full), size=n_train, replace=True)
                X_train = X_full.iloc[idx].reset_index(drop=True)
                y_train = y_full.iloc[idx].reset_index(drop=True)
            else:
                X_train, _, y_train, _ = train_test_split(
                    X_full, y_full, test_size=test_size, random_state=seed
                )
                _, X_test, _, y_test = train_test_split(
                    X_test, y_test, test_size=test_size, random_state=seed
                )

            reg = clone(model)
            if 'random_state' in reg.get_params():
                reg.set_params(random_state=seed)

            reg.fit(X_train, y_train)
            y_pred = reg.predict(X_test)

            mse = mean_squared_error(y_test, y_pred)
            r2_scores.append(r2_score(y_test, y_pred))
            mse_scores.append(mse)
            rmse_scores.append(np.sqrt(mse))
            mae_scores.append(mean_absolute_error(y_test, y_pred))

        results.append({
            'Model': name,
            'R2 Mean': np.mean(r2_scores),
            'R2 Std': _std(r2_scores),
            'MSE Mean': np.mean(mse_scores),
            'MSE Std': _std(mse_scores),
            'RMSE Mean': np.mean(rmse_scores),
            'RMSE Std': _std(rmse_scores),
            'MAE Mean': np.mean(mae_scores),
            'MAE Std': _std(mae_scores),
            'R2 (Mean ± SD)': f"{np.mean(r2_scores):.4f} ± {_std(r2_scores):.4f}",
            'MSE (Mean ± SD)': f"{np.mean(mse_scores):.4f} ± {_std(mse_scores):.4f}",
            'RMSE (Mean ± SD)': f"{np.mean(rmse_scores):.4f} ± {_std(rmse_scores):.4f}",
            'MAE (Mean ± SD)': f"{np.mean(mae_scores):.4f} ± {_std(mae_scores):.4f}",
        })

    return pd.DataFrame(results).sort_values(by='R2 Mean', ascending=False)


In [16]:
print('TRTR (Train Real, Test Real) - 80% train / 20% holdout')
trtr_results = evaluate_regression_models(
    train_df=train_real,
    test_df=test_real,
    label_col=target_col,
    models=regressors,
    seeds=EVAL_SEEDS,
    use_holdout=True,
    schema_df=train_real,
)
display(trtr_results[['Model', 'R2 (Mean ± SD)', 'MSE (Mean ± SD)', 'RMSE (Mean ± SD)', 'MAE (Mean ± SD)']])

all_comparisons = []
for synth_name in GENERATORS_TO_EVAL:
    if synth_name not in synthetic_datasets:
        print(f'Skipping {synth_name} - no synthetic dataset')
        continue

    print(f'{synth_name} - TSTR (train on synthetic, test on 20% holdout)')
    tstr_results = evaluate_regression_models(
        train_df=synthetic_datasets[synth_name],
        test_df=test_real,
        label_col=target_col,
        models=regressors,
        seeds=EVAL_SEEDS,
        use_holdout=True,
        schema_df=train_real,
    )
    display(tstr_results[['Model', 'R2 (Mean ± SD)', 'MSE (Mean ± SD)', 'RMSE (Mean ± SD)', 'MAE (Mean ± SD)']])

    comparison = trtr_results.merge(tstr_results, on='Model', suffixes=('_TRTR', '_TSTR'))
    comparison['R2_Drop'] = comparison['R2 Mean_TRTR'] - comparison['R2 Mean_TSTR']
    comparison['MSE_Increase'] = comparison['MSE Mean_TSTR'] - comparison['MSE Mean_TRTR']
    comparison['RMSE_Increase'] = comparison['RMSE Mean_TSTR'] - comparison['RMSE Mean_TRTR']
    comparison['MAE_Increase'] = comparison['MAE Mean_TSTR'] - comparison['MAE Mean_TRTR']
    comparison['Synthetic_Model'] = synth_name
    all_comparisons.append(comparison)

combined_comparison = pd.concat(all_comparisons, ignore_index=True)
summary = (
    combined_comparison
    .groupby('Synthetic_Model', as_index=False)[['R2_Drop', 'MSE_Increase', 'RMSE_Increase', 'MAE_Increase']]
    .mean()
    .sort_values('R2_Drop')
)

display(summary)


TRTR (Train Real, Test Real) - 80% train / 20% holdout


,Model,R2 (Mean ± SD),MSE (Mean ± SD),RMSE (Mean ± SD),MAE (Mean ± SD)
0,LinearRegression,0.9670 ± 0.0099,6.0008 ± 1.7998,2.4226 ± 0.3827,0.6783 ± 0.1185
2,Lasso,0.9662 ± 0.0105,6.1598 ± 1.9156,2.4531 ± 0.3976,0.8192 ± 0.1148
3,ElasticNet,0.9647 ± 0.0097,6.4218 ± 1.7673,2.5107 ± 0.3625,1.0648 ± 0.0960
8,ExtraTrees,0.9616 ± 0.0119,6.9949 ± 2.1666,2.6126 ± 0.4334,0.7496 ± 0.1430
7,RandomForest,0.9601 ± 0.0095,7.2606 ± 1.7212,2.6764 ± 0.3288,1.1706 ± 0.0908
1,Ridge,0.9576 ± 0.0090,7.7189 ± 1.6444,2.7633 ± 0.3043,1.4837 ± 0.0875
6,DecisionTree,0.9509 ± 0.0182,8.9375 ± 3.3182,2.9440 ± 0.5478,0.9005 ± 0.2423
9,GradientBoost,0.9374 ± 0.0102,11.3970 ± 1.8639,3.3655 ± 0.2792,2.5092 ± 0.0973
5,KNN,0.0720 ± 0.0648,168.9335 ± 11.8018,12.9902 ± 0.4571,10.2440 ± 0.3535
4,SVR_RBF,0.0024 ± 0.0236,181.5960 ± 4.2918,13.4749 ± 0.1571,10.5663 ± 0.0347


Skipping CTGAN - no synthetic dataset
Skipping CopulaGAN - no synthetic dataset
Skipping TVAE - no synthetic dataset
Skipping GaussianCopula - no synthetic dataset
TabDDPM - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean ± SD),MSE (Mean ± SD),RMSE (Mean ± SD),MAE (Mean ± SD)
9,GradientBoost,-0.1657 ± 0.1405,212.2050 ± 25.5835,14.5445 ± 0.8573,11.9610 ± 0.7643
7,RandomForest,-0.2707 ± 0.3020,231.3137 ± 54.9683,15.1234 ± 1.6981,12.2785 ± 1.5110
5,KNN,-1.3672 ± 0.3309,430.9051 ± 60.2417,20.7121 ± 1.4585,16.9371 ± 1.2358
1,Ridge,-1.4366 ± 0.7676,443.5501 ± 139.7329,20.8522 ± 3.1158,16.7419 ± 3.1064
4,SVR_RBF,-1.8802 ± 0.6392,524.2924 ± 116.3512,22.7309 ± 2.9055,18.9124 ± 2.4793
3,ElasticNet,-1.9900 ± 0.8749,544.2854 ± 159.2541,23.1220 ± 3.2758,18.4525 ± 3.2203
2,Lasso,-2.9910 ± 0.9210,726.5077 ± 167.6491,26.7910 ± 3.1181,21.1152 ± 3.2435
0,LinearRegression,-3.1316 ± 0.9720,752.0906 ± 176.9308,27.2515 ± 3.2398,21.4573 ± 3.3406
8,ExtraTrees,-3.1443 ± 1.2627,754.4138 ± 229.8532,27.1506 ± 4.3791,23.1335 ± 4.3126
6,DecisionTree,-4.1550 ± 1.1521,938.3874 ± 209.7295,30.4561 ± 3.4660,26.9491 ± 3.6196


ForestDiffusion - TSTR (train on synthetic, test on 20% holdout)


,Model,R2 (Mean ± SD),MSE (Mean ± SD),RMSE (Mean ± SD),MAE (Mean ± SD)
8,ExtraTrees,0.8012 ± 0.0160,36.1972 ± 2.9104,6.0119 ± 0.2444,4.3154 ± 0.1542
7,RandomForest,0.7968 ± 0.0122,36.9947 ± 2.2220,6.0799 ± 0.1815,4.4251 ± 0.1426
9,GradientBoost,0.7778 ± 0.0073,40.4471 ± 1.3368,6.3590 ± 0.1042,4.6216 ± 0.0813
6,DecisionTree,0.7522 ± 0.0228,45.1128 ± 4.1541,6.7102 ± 0.3097,4.8414 ± 0.2208
1,Ridge,0.7337 ± 0.0134,48.4819 ± 2.4371,6.9609 ± 0.1757,5.1405 ± 0.1216
3,ElasticNet,0.7266 ± 0.0158,49.7674 ± 2.8762,7.0519 ± 0.2042,5.2355 ± 0.1424
2,Lasso,0.7157 ± 0.0177,51.7490 ± 3.2310,7.1905 ± 0.2241,5.3715 ± 0.1443
0,LinearRegression,0.7019 ± 0.0216,54.2575 ± 3.9358,7.3616 ± 0.2670,5.5413 ± 0.2218
5,KNN,0.0223 ± 0.0438,177.9793 ± 7.9718,13.3379 ± 0.2993,10.2811 ± 0.2695
4,SVR_RBF,-0.0347 ± 0.0260,188.3429 ± 4.7270,13.7228 ± 0.1718,10.5484 ± 0.0567


,Synthetic_Model,R2_Drop,MSE_Increase,RMSE_Increase,MAE_Increase
0,ForestDiffusion,0.174642,31.790902,3.257335,3.013572
1,TabDDPM,2.827230,514.653042,18.052099,15.775240


In [17]:
output_file = 'TRTR_TSTR_results_regression.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    quality_df.to_excel(writer, sheet_name='Quality_Metrics', index=False)
    trtr_results.to_excel(writer, sheet_name='TRTR_Results', index=False)
    combined_comparison.to_excel(writer, sheet_name='All_Comparisons', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)
    for synth_name in GENERATORS_TO_EVAL:
        if synth_name in combined_comparison['Synthetic_Model'].values:
            synth_results = combined_comparison[combined_comparison['Synthetic_Model'] == synth_name]
            synth_results.to_excel(writer, sheet_name=synth_name[:31], index=False)

print(f'Results saved to: {output_file}')

Results saved to: TRTR_TSTR_results_regression.xlsx
